# PRISMATIC Workshop: Initializing FATES from NEON Remote Sensing

This notebook walks through the PRISMATIC pipeline — from downloading raw NEON field and airborne data to generating plant functional type (PFT) initial conditions for the FATES dynamic vegetation model.

**Site:** TEAK (Lower Teakettle), California  
**Year:** 2021  
**Model target:** FATES (Functionally Assembled Terrestrial Ecosystem Simulator)

---

## Background

currently a ppt. convert to canva link

## Setup

In [ ]:
import sys, os
sys.path.insert(0, '..')

from hydra import initialize, compose
from omegaconf import OmegaConf

from utils.utils import build_cache_site, force_rerun
from initialize.inventory import download_veg_structure_data, download_trait_table, prep_veg_structure
from initialize.plots import download_polygons, prep_polygons
from initialize.lidar import download_lidar, download_aop_bbox, normalize_laz, clip_lidar_by_plots
from initialize.lad import prep_lad
from initialize.biomass import prep_biomass
from initialize.hyperspectral import (download_hyperspectral, correct_flightlines,
                                      prep_manual_training_data, prep_aop_imagery,
                                      extract_spectra_from_polygon, train_pft_classifier)
from initialize.generate_initial_conditions import generate_initial_conditions

with initialize(config_path='conf', version_base=None):
    cfg = compose(config_name='config')

site = 'TEAK'
year = '2021'
site_cfg = cfg.sites.run[site][year]
year_aop = site_cfg.year_aop

print(OmegaConf.to_yaml(site_cfg))

# --- Values from the Hydra config (the same ones main.py pulls out) ---
data_raw_aop_path = cfg.paths.data_raw_aop_path
data_raw_inv_path = cfg.paths.data_raw_inv_path
data_int_path     = cfg.paths.data_int_path
data_final_path   = cfg.paths.data_final_path

ic_type          = cfg.others.ic_type
hs_type          = cfg.others.hs_type
month_window     = cfg.others.month_window
n_plots          = cfg.others.n_plots
plot_length      = cfg.others.plot_length
ntree            = cfg.others.ntree
min_distance     = cfg.others.min_distance
use_tiles_w_veg  = cfg.others.use_tiles_w_veg
randomMinSamples = cfg.others.randomMinSamples
aggregate_from_1m_to_2m_res = cfg.others.aggregate_from_1m_to_2m_res
independentValidationSet    = cfg.others.independentValidationSet
pcaInsteadOfWavelengths     = cfg.others.pcaInsteadOfWavelengths
multisite        = cfg.others.multisite
coords_bbox      = cfg.others.coords_bbox
neon_trait_link  = cfg.others.neon_trait_table.neon_trait_link

use_case = "train"

# step(): run a pipeline function only if its outputs aren't already on disk.
# This mirrors main.py's force_rerun/cache wiring, so re-running a cell skips work
# that is already done and returns the existing output path(s) instead.
rerun_status = {k: bool(v) for k, v in site_cfg.force_rerun.items()}

def step(fn, **kwargs):
    cache = build_cache_site(site=site, year_inventory=year, year_aop=year_aop,
                             data_raw_aop_path=data_raw_aop_path,
                             data_raw_inv_path=data_raw_inv_path,
                             data_int_path=data_int_path,
                             hs_type=hs_type, coords_bbox=coords_bbox)
    return force_rerun(cache, force=rerun_status)(fn)(**kwargs)

---

## 1. Downloading Field Inventory Data

**Functions:** `download_veg_structure_data`, `download_polygons`, `download_trait_table`

Access NEON's ground-based forest inventory (stem locations, species IDs, DBH) and plot polygon boundaries via the NEON API — the ecological ground truth that anchors everything downstream.

In [ ]:
# Download stem-level vegetation structure data (species, DBH, height, location)
veg_structure_path, sampling_effort_path = step(
    download_veg_structure_data, site=site, data_path=data_raw_inv_path)

# Download NEON plot boundary polygons
neon_plots_path = step(download_polygons, data_path=data_raw_inv_path)

# Download the NEON trait table (used later for allometric biomass equations)
trait_table_path = step(
    download_trait_table, download_link=neon_trait_link, data_path=data_raw_inv_path)

---

## 2. Downloading Airborne Remote Sensing Data

**Functions:** `download_lidar`, `download_hyperspectral` (or `download_aop_bbox` for a spatial subset)

Retrieve NEON AOP (Airborne Observation Platform) LiDAR point clouds and hyperspectral imagery — the two remote sensing data streams that drive PFT classification at landscape scale.

In [ ]:
# Download AOP LiDAR (.laz) + hyperspectral imagery for the site's flight year.
# If coords_bbox is set in conf, both are fetched clipped to that box in one call instead.
if not coords_bbox:
    laz_path, tif_path = step(
        download_lidar, site=site, year=year_aop,
        lidar_path=data_raw_aop_path, use_tiles_w_veg=use_tiles_w_veg)
    hs_path = step(
        download_hyperspectral, site=site, year=year_aop,
        data_raw_aop_path=data_raw_aop_path, hs_type=hs_type)
else:
    hs_path, laz_path, tif_path = step(
        download_aop_bbox, site=site, year=year_aop,
        path=data_raw_aop_path, hs_type=hs_type, coords_bbox=coords_bbox)

---

## 3. Processing Field Inventory into Training Labels

**Functions:** `prep_veg_structure`, `prep_polygons`

Clean and filter stem measurements to the target year, assign PFT labels to individual plants, and partition NEON plots into spatial units suitable for linking to remote sensing pixels.

> We will classify PFTs from NEON's airborne LiDAR and hyperspectral data and train the model on ground-based inventories.

In [ ]:
# Filter stems to the target year window and assign PFT labels
inventory_file_path, sampling_effort_path = step(
    prep_veg_structure, site=site, year_inv=year, year_aop=year_aop,
    data_path=data_raw_inv_path, month_window=month_window)

# Partition plot polygons into spatial subunits for linking to remote sensing
partitioned_plots_path = step(
    prep_polygons, input_data_path=neon_plots_path,
    sampling_effort_path=sampling_effort_path, inventory_path=inventory_file_path,
    site=site, year=year, output_data_path=data_int_path)

---

## 4. Processing LiDAR: Normalization and Clipping

**Functions:** `normalize_laz`, `clip_lidar_by_plots`

Height-normalize raw LiDAR point clouds (removing terrain so heights reflect canopy, not topography), then clip to inventory plot boundaries to align 3D structure data with field observations.

In [ ]:
# Subtract the digital terrain model so z reflects canopy height above ground
normalized_laz_path = step(
    normalize_laz, laz_path=laz_path, site=site, year=year, output_path=data_int_path)

# Clip normalized point clouds to the partitioned plot boundaries
clipped_laz_path = step(
    clip_lidar_by_plots, laz_path=normalized_laz_path, tif_path=tif_path,
    site_plots_path=partitioned_plots_path, site=site, year=year,
    output_laz_path=data_int_path, end_result=True)

---

## 5. Deriving Canopy Structure: Leaf Area Density Profiles

**Function:** `prep_lad`

Compute vertical leaf area density (LAD) profiles from normalized LiDAR within each plot, stratified by size class — a key structural descriptor of canopy layering and vegetation density.

### FATES Cohort Structure

The LAD profiles map directly onto FATES cohort height classes. Each cohort in FATES represents plants of similar height competing for light — the canopy layering captured here defines the initial vertical structure of the simulated forest.

![FATES cohort diagram placeholder](docs/fates_cohorts_placeholder.png)

> We will classify PFTs from NEON's airborne LiDAR and hyperspectral data and train the model on ground-based inventories.

In [ ]:
# Compute leaf area density (LAD) profiles per plot, stratified by PFT size class
prep_lad_path = step(
    prep_lad, laz_path=clipped_laz_path, inventory_path=inventory_file_path,
    site=site, year=year, output_path=data_int_path, use_case=use_case)

---

## 6. Estimating Biomass

**Function:** `prep_biomass`

Apply allometric equations to stem measurements (using a NEON trait table) to estimate above-ground biomass per plant functional type per plot — providing a carbon stock summary alongside structural data.

In [ ]:
# Estimate above-ground biomass per stem via species-specific allometry (NEON trait table)
biomass_path = step(
    prep_biomass, data_path=inventory_file_path, site_plots_path=partitioned_plots_path,
    sampling_effort_path=sampling_effort_path, site=site, year=year,
    data_int_path=data_int_path, neon_trait_table_path=trait_table_path,
    end_result=False)

---

## 7. Preparing Hyperspectral Imagery

**Functions:** `correct_flightlines`, `prep_aop_imagery`

Apply BRDF and topographic corrections to hyperspectral flightlines, then stack hyperspectral bands with LiDAR-derived rasters into a single multi-layer image ready for pixel-level classification.

> We will classify PFTs from NEON's airborne LiDAR and hyperspectral data and train the model on ground-based inventories.

In [ ]:
# These remote-sensing steps run for every ic_type except "field_inv_plots".
if ic_type != "field_inv_plots":
    # BRDF + topographic correction is only needed for flightline hyperspectral
    if hs_type == "flightline":
        step(correct_flightlines, site=site, year_inv=year, year_aop=year_aop,
             data_raw_aop_path=data_raw_aop_path, data_int_path=data_int_path)

    # Stack hyperspectral bands + LiDAR-derived rasters into one multi-layer image
    stacked_aop_path = step(
        prep_aop_imagery, site=site, year=year, hs_type=hs_type,
        hs_path=hs_path, tif_path=tif_path, data_int_path=data_int_path,
        use_tiles_w_veg=use_tiles_w_veg)

---

## 8. Building the Training Dataset

**Functions:** `prep_manual_training_data`, `extract_spectra_from_polygon`

Overlay inventory-derived crown polygons onto the stacked AOP imagery to extract per-pixel spectral signatures, producing a labeled training table (spectral features + PFT class) for the classifier.

In [ ]:
if ic_type != "field_inv_plots":
    # Build labelled training crowns from the inventory + biomass
    training_crown_shp_path = step(
        prep_manual_training_data, site=site, year=year,
        data_raw_inv_path=data_raw_inv_path, data_int_path=data_int_path,
        biomass_path=biomass_path)

    # Extract per-pixel spectra within each crown polygon and attach PFT labels
    training_spectra_csv_path = step(
        extract_spectra_from_polygon, site=site, year=year,
        shp_path=training_crown_shp_path, data_int_path=data_int_path,
        data_final_path=data_final_path, stacked_aop_path=stacked_aop_path,
        use_case=use_case, aggregate_from_1m_to_2m_res=aggregate_from_1m_to_2m_res,
        ic_type=ic_type)

---

## 9. Training the PFT Classifier

**Function:** `train_pft_classifier`

Train a Random Forest model on the labeled spectral training data (optionally using PCA instead of raw wavelengths) and evaluate accuracy — producing a model that can predict PFT identity for any pixel in the scene.

In [ ]:
# Train the Random Forest PFT classifier on the labelled spectra; report held-out accuracy
sites = [site]
rf_model_path = step(
    train_pft_classifier, sites=sites, data_int_path=data_int_path,
    pcaInsteadOfWavelengths=pcaInsteadOfWavelengths, ntree=ntree,
    randomMinSamples=randomMinSamples, independentValidationSet=independentValidationSet)

---

## 10. Generating Initial Conditions for FATES

**Function:** `generate_initial_conditions`

Apply the trained classifier wall-to-wall across the site, then aggregate pixel-level PFT predictions and biomass into FATES cohort and patch files — the end product that initializes a land surface model with spatially-informed vegetation structure.

> Here we have now classified plant functional composition (height class, PFT) across a single tile for a NEON site and year.

In [ ]:
# Classify wall-to-wall, aggregate to cohorts/patches, and write FATES IC files
use_case = "predict"
ic_type_path = os.path.join(data_final_path, site, year, ic_type)
os.makedirs(ic_type_path, exist_ok=True)

cohort_path, patch_path = step(
    generate_initial_conditions, site=site, year_inv=year, year_aop=year_aop,
    data_raw_aop_path=data_raw_aop_path, data_int_path=data_int_path,
    data_final_path=data_final_path, rf_model_path=rf_model_path,
    stacked_aop_path=os.path.join(data_int_path, site, year, 'stacked_aop'),
    biomass_path=os.path.join(data_int_path, site, year, 'biomass/pp_veg_structure_IND_IBA_IAGB_live.csv'),
    use_case=use_case, ic_type=ic_type, ic_type_path=ic_type_path,
    n_plots=n_plots, min_distance=min_distance, plot_length=plot_length,
    aggregate_from_1m_to_2m_res=aggregate_from_1m_to_2m_res,
    pcaInsteadOfWavelengths=pcaInsteadOfWavelengths, multisite=multisite)

print('cohort file:', cohort_path)
print('patch file:', patch_path)

---

## 11. Where We're Going: FATES Simulations

The initial conditions we just generated feed directly into FATES simulations. Below are example outputs from FATES runs initialized with PRISMATIC data — showing how the spatially-informed PFT structure and biomass translate into simulated forest dynamics over time.

*(Simulation figures go here)*